# Topic: Batch vs Real-Time Prediction

## Definition (30-second explanation)
*   **Analogy:** Batch prediction is like a catering company preparing 1,000 identical boxed lunches overnight—it is highly efficient, but you get what is in the box. Real-time prediction is a short-order cook making a specific meal exactly when you order it—it is fresh and customized to your current mood, but much more expensive to staff.
*   It is the fundamental architectural decision of whether to pre-compute predictions offline and store them, or compute them on-the-fly when a user request arrives.

## Why Interviewers Ask This
*   Choosing real-time inference when batch would suffice is a classic junior engineer mistake that wastes massive amounts of infrastructure budget. Interviewers want to see if you possess the business acumen to balance latency requirements against cloud costs.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Computing heavy ML models instantly for millions of concurrent users is often mathematically or financially impossible. 
*   **The Mechanism:** 
    *   **Batch:** Uses big data tools (Spark, Airflow) to process millions of records at scheduled intervals (e.g., nightly), saving the output to a database for later retrieval.
    *   **Real-Time:** Wraps the model in a REST API or gRPC service. The model loads into memory on an always-on server and waits to score individual user requests in <100ms.
*   **The Trade-off:** Real-time gives you the freshest possible predictions (incorporating data from the user's current session) but costs significantly more and is harder to maintain than batch processing. 

## When to Use
*   **Batch:** Use when predictions do not need to be instant (e.g., generating churn scores, nightly ad targeting, or daily email recommendations) and features require expensive database joins.
*   **Real-Time:** Use when the user experience depends on an instant response (e.g., credit card fraud check at checkout) or when input features are only generated at request time (e.g., the user's last click).

## Advantages
*   **Batch:** Extremely high throughput (millions of rows per run), low cost because servers only run when needed, and highly resilient (if it fails, just rerun the job).
*   **Real-Time:** Low latency and highly accurate because it uses "fresh" features representing the current state of the world.

## Limitations
*   **Batch:** Predictions are stale. If a user's behavior changes drastically at 9:00 AM, the nightly batch model won't know until the next day.
*   **Real-Time:** Infrastructure is always-on (expensive), throughput is bottlenecked by concurrency, and failure recovery is complex (you cannot easily replay dropped live requests).

## Common Comparisons
*   **Lambda Architecture (Hybrid):** Combines both. You pre-compute massive "candidate sets" (e.g., 500 possible movies to watch) using a Batch layer overnight, and then use a lightweight Real-Time speed layer to instantly re-rank the top 10 movies based on the user's session clicks.

## Common Interview Traps
*   **Over-engineering Real-Time:** Candidates often say they will deploy a real-time Kafka stream for every problem. *Trap:* Always validate that real-time freshness is actually required by the business before recommending expensive serving infrastructure.

## Python Syntax
```python
# Real-Time API Prediction Class Setup
import time
import joblib
import numpy as np

class RealtimePredictor:
    def __init__(self, model_path):
        # 1. Load model once at startup to avoid disk I/O on every request
        self.model = joblib.load(model_path)

    def predict(self, features: dict) -> dict:
        start = time.time()
        
        # 2. Parse features into numpy array for the model
        X = np.array([[features['age'], features['income'], features['score']]])
        
        # 3. Predict probability for the positive class
        prob = self.model.predict_proba(X)[0][1]
        
        # 4. Measure latency internally for monitoring
        latency_ms = (time.time() - start) * 1000
        
        return {
            'probability': round(float(prob), 4),
            'prediction': int(prob >= 0.5),
            'latency_ms': round(latency_ms, 2)
        }
```

## 45-Second Interview Answer
"Choosing between batch and real-time inference comes down to balancing latency, cost, and feature freshness. Batch prediction is high-throughput and cost-effective, running on a schedule to pre-compute things like nightly churn scores. Real-time prediction is low-latency, running on always-on servers for use cases like fraud detection where you need instant results based on fresh data. Because real-time is vastly more expensive and complex, I always validate the business requirement before building it, often opting for a hybrid Lambda architecture where possible."

## Practice Questions:

### Q1:
**Question:**
A customer asks you to build a recommendation system. How would you decide between batch and real-time inference?

**Ideal Interview Answer:**
"I evaluate three factors: Freshness, Scale, and Feature Availability. 
1. **Freshness:** If recommendations must adapt to what the user clicked 5 seconds ago, we need real-time. If daily updates are fine, batch wins.
2. **Scale:** If we must score 100 million users, doing this nightly via a batch Spark job is efficient and cheap.
3. **Feature Availability:** If the model relies on in-session context, batch physically cannot work. 
Ideally, I would recommend a hybrid Lambda architecture: we pre-compute user embeddings and candidate items in batch overnight, but perform the final ranking in real-time to incorporate live session data."

**Common Mistakes Candidates Make:**
*   Immediately suggesting a complex real-time system without validating if the business actually needs instant recommendations. Real-time serving is much harder to maintain.

**Likely Interviewer Follow-up:**
"Can you walk me through the system design of that Hybrid (Lambda) architecture? What exact data moves between the batch layer and the real-time layer?"

(Answer: 
"The system design acts as a two-stage funnel:
1. **Batch Layer (Offline Retrieval):** Overnight, heavy processing jobs (like Spark) analyze a user's entire history to compute a broad 'Candidate Set' (e.g., their top 200 relevant items). This list, along with heavy pre-computed feature embeddings, is saved to a fast-read NoSQL database like Redis.
2. **Real-Time Layer (Online Re-ranking):** When the user opens the app, the real-time API instantly fetches those 200 candidates from Redis. It then passes them through a lightweight 're-ranking' model that adjusts the final order based on live, in-session features (e.g., what the user clicked 2 seconds ago). 

This architecture moves the heavy computational lifting offline, while the real-time layer only handles lightweight sorting to keep the final output fresh.")

### Q2: Lambda Architecture for Recommendations
**Question:** 
Describe the data flow for a hybrid (Lambda) architecture for a video recommendation feed. What happens in the Batch Layer vs. the Real-Time Layer?

**Answer:**
"This system uses a two-stage funnel: Candidate Generation (Batch) and Re-ranking (Real-Time).
1. **Batch Layer (Candidate Generation):** Overnight, heavy processing jobs run complex algorithms (collaborative filtering, content embeddings, and MMR for diversity) on the user's entire historical watch history. It whittles millions of videos down to a 'Candidate Set' of 500 highly relevant video IDs. These 500 IDs and their embeddings are saved to a low-latency cache like Redis.
2. **Real-Time Layer (Re-ranking):** When the user opens the app, the real-time API has a strict 100ms latency budget. It instantly fetches the 500 candidate IDs from Redis and passes them through a lightweight ranking model. This model sorts the videos into a final top-50 feed by applying real-time context (e.g., time of day, device type, or what the user clicked two seconds ago)."

**Common Interview Traps:**
* Attempting to run heavy collaborative filtering or vector similarity searches across the entire database of millions of videos in real-time. (This will cause timeouts).
* Worrying about where the actual `mp4` video files are stored. The ML pipeline only cares about processing Metadata, Embeddings, and Video IDs.